In [ ]:
!pip install --upgrade ipywidgets
!pip install --upgrade jupyter


In [1]:
import pandas as pd
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
file_path = "merged_recipes.xlsx"  # Update path to your uploaded file
recipe_df = pd.read_excel(file_path)

# Preprocess recipe data
recipe_df['Combined Text'] = recipe_df['Ingredients'] + " " + recipe_df['Cuisine Region'] + " " + recipe_df['Recipe']

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to encode text using BERT
def encode_text(texts):
    encoded_inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=512)
    with torch.no_grad():
        model_outputs = model(**encoded_inputs)
    embeddings = model_outputs.last_hidden_state.mean(dim=1)  # Pooling over tokens
    return embeddings

# Encode all recipes in the dataset
recipe_embeddings = encode_text(recipe_df['Combined Text'].tolist())

# Function to predict recipe based on user ingredients
def predict_recipe(user_ingredients):
    # Encode user input
    user_input_embedding = encode_text([user_ingredients])
    
    # Calculate cosine similarity between user input and all recipe embeddings
    similarities = cosine_similarity(user_input_embedding.cpu().numpy(), recipe_embeddings.cpu().numpy())
    
    # Get the index of the most similar recipe
    most_similar_idx = similarities.argmax()
    
    # Retrieve the most similar recipe
    most_similar_recipe = recipe_df.iloc[most_similar_idx]
    return most_similar_recipe['Recipe'], most_similar_recipe['Ingredients'], most_similar_recipe['Cuisine Region']

# Example usage
user_ingredients = "chicken tomatoes garlic onions"
recipe, ingredients, cuisine = predict_recipe(user_ingredients)

print("Recommended Recipe:", recipe)
print("Ingredients:", ingredients)
print("Cuisine Region:", cuisine)


C:\Users\user\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 3655065600 bytes.